# Churn modeling: SaaS usage analytics

This notebook trains and evaluates churn prediction models, generates SHAP explanations, and performs cost-benefit analysis for at-risk account identification.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_and_prepare
from src.model import train_and_evaluate

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## Load and prepare data

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_prepare("../data/saas_usage.csv")
print(f"\nFeature count: {len(feature_names)}")
print(f"Feature names: {feature_names}")

## Train models and evaluate

This trains Logistic Regression, Random Forest, XGBoost, and LightGBM with GridSearchCV. It also generates SHAP plots and performs at-risk account cost-benefit analysis.

In [ ]:
import os
os.chdir("..")
results = train_and_evaluate(X_train, X_test, y_train, y_test, feature_names)

## Review saved outputs

In [ ]:
# Model comparison
comparison = pd.read_csv("outputs/model_comparison.csv", index_col=0)
print("Model comparison:")
print(comparison.to_string())
print(f"\nBest model: {comparison['auc_roc'].idxmax()} (AUC = {comparison['auc_roc'].max():.4f})")

In [ ]:
# Display saved plots
from IPython.display import Image, display

for plot_name in ["roc_curves.png", "confusion_matrices.png", "feature_importance.png",
                  "shap_summary.png", "shap_waterfall.png", "at_risk_analysis.png"]:
    path = f"outputs/{plot_name}"
    if os.path.exists(path):
        print(f"\n--- {plot_name} ---")
        display(Image(filename=path, width=700))

## Summary

Key takeaways from the churn modeling:

1. **LightGBM achieved the best AUC-ROC** among the four models tested, making it the selected production model
2. **Daily logins and feature adoption** are the top SHAP features driving churn predictions
3. **Plan tier** (especially Free) is a strong predictor, confirming the EDA findings
4. **Support ticket volume** and **NPS score** provide complementary churn signals beyond pure engagement metrics
5. **At-risk account scoring** with optimized threshold enables targeted CS outreach that maximizes net revenue saved
6. **Cost-benefit analysis** shows that proactive outreach to high-risk accounts generates significant net savings even after accounting for outreach costs